In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Load both the training and test datasets
train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
# Separate the target variable from the training data
y_train = train_df['SalePrice']
train_features = train_df.drop('SalePrice', axis=1)

# Store the test IDs for the final submission file
test_ids = test_df['Id']
test_features = test_df.drop('Id', axis=1)

# Combine training and test features for unified preprocessing
combined_features = pd.concat([train_features, test_features], ignore_index=True)

In [ ]:
# --- Handling Missing Values ---

# Drop columns with more than 80% missing values
missing_values = combined_features.isnull().sum() / len(combined_features)
cols_to_drop = missing_values[missing_values > 0.8].index
combined_features = combined_features.drop(columns=cols_to_drop)

# For numerical features, impute with the mean
numerical_cols = combined_features.select_dtypes(include=np.number).columns
numerical_imputer = SimpleImputer(strategy='mean')
combined_features[numerical_cols] = numerical_imputer.fit_transform(combined_features[numerical_cols])

# For categorical features, impute with a constant value 'None' or the mode
categorical_cols = combined_features.select_dtypes(include=['object']).columns
categorical_imputer = SimpleImputer(strategy='constant', fill_value='None')
combined_features[categorical_cols] = categorical_imputer.fit_transform(combined_features[categorical_cols])

In [ ]:
# --- Feature Engineering ---

# Create a new feature for total square footage
combined_features['TotalSF'] = combined_features['TotalBsmtSF'] + combined_features['1stFlrSF'] + combined_features['2ndFlrSF']

In [ ]:
# --- Encoding Categorical Features ---

# Use One-Hot Encoding for categorical features
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = encoder.fit_transform(combined_features[categorical_cols])

# Create a DataFrame from the encoded features
X_encoded_df = pd.DataFrame(X_encoded, columns=encoder.get_feature_names_out(categorical_cols), index=combined_features.index)

# Combine numerical and encoded categorical features
preprocessed_features = pd.concat([combined_features.select_dtypes(include=np.number), X_encoded_df], axis=1)

In [ ]:
# Split the combined data back into training and test sets
X_train = preprocessed_features.iloc[:len(train_features), :]
X_test = preprocessed_features.iloc[len(train_features):, :]

# Display the final shape of the preprocessed data
print(f"Final shape of preprocessed training data: {X_train.shape}")
print(f"Final shape of preprocessed test data: {X_test.shape}")

In [ ]:
# --- Correlation Heatmap ---
# Select the top 10 most correlated features with 'SalePrice'
correlation_matrix = train_df.corr(numeric_only=True)
top_corr_features = correlation_matrix.nlargest(11, 'SalePrice')['SalePrice'].index
corr_heatmap = train_df[top_corr_features].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr_heatmap, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Top Features with SalePrice')
plt.show()

In [ ]:
# --- Scatter Plots for Key Features ---
# Create scatter plots for the top 3 features (besides SalePrice)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.scatterplot(data=train_df, x='GrLivArea', y='SalePrice')
plt.title('GrLivArea vs. SalePrice')

plt.subplot(1, 3, 2)
sns.scatterplot(data=train_df, x='GarageArea', y='SalePrice')
plt.title('GarageArea vs. SalePrice')

plt.subplot(1, 3, 3)
sns.scatterplot(data=train_df, x='TotalBsmtSF', y='SalePrice')
plt.title('TotalBsmtSF vs. SalePrice')

plt.tight_layout()
plt.show()

In [ ]:
# --- Box Plots for Key Categorical Features ---
# Use box plots to visualize relationship between categorical features and SalePrice
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
sns.boxplot(data=train_df, x='OverallQual', y='SalePrice')
plt.title('OverallQual vs. SalePrice')

plt.subplot(1, 2, 2)
sns.boxplot(data=train_df, x='Neighborhood', y='SalePrice')
plt.xticks(rotation=90)
plt.title('Neighborhood vs. SalePrice')

plt.tight_layout()
plt.show()

In [ ]:
# --- Model Selection and Training ---
# Use Lasso as a preliminary baseline model
lasso_model = Lasso(alpha=0.001, max_iter=50000)
lasso_model.fit(X_train, y_train)

In [ ]:
# --- Model Prediction ---
# Predict on the training data to calculate performance metrics
y_train_pred = lasso_model.predict(X_train)

In [ ]:
# Calculate performance metrics on the training set
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

print("--- Preliminary Model Performance (Lasso Regression) on Training Data ---")
print(f"Root Mean Squared Error (RMSE): ${rmse_train:,.2f}")
print(f"Mean Absolute Error (MAE): ${mae_train:,.2f}")
print(f"R-squared (R²): {r2_train:.4f}")

In [ ]:
# Predict on the test data
test_predictions = lasso_model.predict(X_test)

# --- Submission File Generation ---
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_predictions
})

# Save the submission file to a CSV
submission.to_csv('submission.csv', index=False)

print("\nSubmission file 'submission.csv' created successfully.")

In [ ]:
"""
Import additional libraries for advanced modeling and analysis
"""
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
"""
Split training data into train/validation for robust evaluation
"""
# Create a validation set from training data
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Training split shape: {X_train_split.shape}")
print(f"Validation split shape: {X_val_split.shape}")
print(f"Full training shape: {X_train.shape}")

In [ ]:
"""
Compare multiple regression models with cross-validation
"""
# Initialize multiple models
models = {
    'Lasso': Lasso(alpha=0.001, max_iter=50000),
    'Ridge': Ridge(alpha=10.0),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=50000),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, verbose=-1, force_col_wise=True)
}

# Perform cross-validation for each model
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("=" * 80)
print("CROSS-VALIDATION RESULTS (5-Fold)")
print("=" * 80)

for name, model in models.items():
    # Calculate cross-validation RMSE scores
    cv_scores = -cross_val_score(model, X_train, y_train, cv=kfold, 
                                  scoring='neg_root_mean_squared_error', n_jobs=-1)
    results[name] = {
        'cv_rmse_mean': cv_scores.mean(),
        'cv_rmse_std': cv_scores.std()
    }
    print(f"{name:20s} | RMSE: ${cv_scores.mean():>10,.2f} (+/- ${cv_scores.std():>8,.2f})")

print("=" * 80)

In [ ]:
"""
Visualize cross-validation results across models
"""
fig, ax = plt.subplots(figsize=(12, 6))

model_names = list(results.keys())
means = [results[name]['cv_rmse_mean'] for name in model_names]
stds = [results[name]['cv_rmse_std'] for name in model_names]

colors = plt.cm.viridis(np.linspace(0, 0.9, len(model_names)))
bars = ax.bar(model_names, means, yerr=stds, capsize=5, color=colors, alpha=0.7, edgecolor='black')

ax.set_ylabel('RMSE ($)', fontsize=12, fontweight='bold')
ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison (5-Fold CV)', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Add value labels on bars
for bar, mean in zip(bars, means):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${mean:,.0f}',
            ha='center', va='bottom', fontsize=9)

plt.show()

In [ ]:
"""
Train all models and evaluate on validation data
"""
validation_results = {}

print("=" * 80)
print("VALIDATION SET PERFORMANCE")
print("=" * 80)

for name, model in models.items():
    # Train on training split
    model.fit(X_train_split, y_train_split)
    
    # Predict on validation split
    y_val_pred = model.predict(X_val_split)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_val_split, y_val_pred))
    mae = mean_absolute_error(y_val_split, y_val_pred)
    r2 = r2_score(y_val_split, y_val_pred)
    
    validation_results[name] = {
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }
    
    print(f"\n{name}:")
    print(f"  RMSE: ${rmse:>10,.2f}")
    print(f"  MAE:  ${mae:>10,.2f}")
    print(f"  R²:   {r2:>10.4f}")

print("=" * 80)

In [ ]:
"""
Analyze feature importance using tree-based models
"""
# Train Random Forest on full training data for feature importance
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Display top 20 features
print("=" * 80)
print("TOP 20 MOST IMPORTANT FEATURES (Random Forest)")
print("=" * 80)
print(feature_importance.head(20).to_string(index=False))
print("=" * 80)

# Visualize top 15 features
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(15)
ax.barh(range(len(top_features)), top_features['importance'], color='steelblue', alpha=0.7)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance Score', fontweight='bold')
ax.set_title('Top 15 Feature Importances', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
"""
Identify outliers in target variable and predictions
"""
# Analyze target variable distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(y_train, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sale Price', fontweight='bold')
axes[0].set_ylabel('Frequency', fontweight='bold')
axes[0].set_title('Sale Price Distribution', fontweight='bold')
axes[0].axvline(y_train.mean(), color='red', linestyle='--', label=f'Mean: ${y_train.mean():,.0f}')
axes[0].axvline(y_train.median(), color='green', linestyle='--', label=f'Median: ${y_train.median():,.0f}')
axes[0].legend()

# Box plot
axes[1].boxplot(y_train, vert=True)
axes[1].set_ylabel('Sale Price', fontweight='bold')
axes[1].set_title('Sale Price Box Plot', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# Log-transformed histogram
log_prices = np.log1p(y_train)
axes[2].hist(log_prices, bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Log(Sale Price)', fontweight='bold')
axes[2].set_ylabel('Frequency', fontweight='bold')
axes[2].set_title('Log-Transformed Sale Price', fontweight='bold')

plt.tight_layout()
plt.show()

# Statistical summary
print("\n" + "=" * 80)
print("TARGET VARIABLE STATISTICS")
print("=" * 80)
print(f"Mean:     ${y_train.mean():>12,.2f}")
print(f"Median:   ${y_train.median():>12,.2f}")
print(f"Std Dev:  ${y_train.std():>12,.2f}")
print(f"Min:      ${y_train.min():>12,.2f}")
print(f"Max:      ${y_train.max():>12,.2f}")
print(f"Skewness: {y_train.skew():>12.4f}")
print(f"Kurtosis: {y_train.kurtosis():>12.4f}")
print("=" * 80)

# Identify outliers using IQR method
Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1
outlier_threshold_low = Q1 - 1.5 * IQR
outlier_threshold_high = Q3 + 1.5 * IQR
outliers = y_train[(y_train < outlier_threshold_low) | (y_train > outlier_threshold_high)]

print(f"\nNumber of outliers detected: {len(outliers)} ({len(outliers)/len(y_train)*100:.2f}%)")
print(f"Outlier thresholds: ${outlier_threshold_low:,.2f} - ${outlier_threshold_high:,.2f}")


In [ ]:
"""
Analyze residuals to identify model issues
"""
# Use the best performing model (from validation results)
best_model_name = min(validation_results.keys(), key=lambda x: validation_results[x]['rmse'])
print(f"Analyzing residuals for best model: {best_model_name}")

best_model = models[best_model_name]
best_model.fit(X_train_split, y_train_split)
y_val_pred = best_model.predict(X_val_split)

# Calculate residuals
residuals = y_val_split - y_val_pred

# Create residual plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Predicted
axes[0, 0].scatter(y_val_pred, residuals, alpha=0.5, color='steelblue')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Sale Price', fontweight='bold')
axes[0, 0].set_ylabel('Residuals', fontweight='bold')
axes[0, 0].set_title('Residuals vs Predicted Values', fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# Residuals distribution
axes[0, 1].hist(residuals, bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Residuals', fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontweight='bold')
axes[0, 1].set_title('Residuals Distribution', fontweight='bold')
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)

# Q-Q Plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot', fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Actual vs Predicted
axes[1, 1].scatter(y_val_split, y_val_pred, alpha=0.5, color='green')
axes[1, 1].plot([y_val_split.min(), y_val_split.max()], 
                [y_val_split.min(), y_val_split.max()], 
                'r--', linewidth=2, label='Perfect Prediction')
axes[1, 1].set_xlabel('Actual Sale Price', fontweight='bold')
axes[1, 1].set_ylabel('Predicted Sale Price', fontweight='bold')
axes[1, 1].set_title('Actual vs Predicted', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nResidual Statistics:")
print(f"Mean Residual: ${residuals.mean():,.2f} (should be close to 0)")
print(f"Std Dev of Residuals: ${residuals.std():,.2f}")

In [ ]:
"""
Create additional engineered features for better performance
"""
# Create a copy for enhanced features
X_train_enhanced = X_train.copy()
X_test_enhanced = X_test.copy()

# We need to work with the combined features to ensure consistency
combined_features_enhanced = combined_features.copy()

# Calculate feature statistics for engineering
print("Creating enhanced features...")

# Age features (if YearBuilt exists in numerical columns)
if 'YearBuilt' in combined_features_enhanced.columns:
    current_year = 2025
    combined_features_enhanced['HouseAge'] = current_year - combined_features_enhanced['YearBuilt']
    if 'YearRemodAdd' in combined_features_enhanced.columns:
        combined_features_enhanced['YearsSinceRemodel'] = current_year - combined_features_enhanced['YearRemodAdd']
        combined_features_enhanced['IsRemodeled'] = (
            combined_features_enhanced['YearRemodAdd'] != combined_features_enhanced['YearBuilt']
        ).astype(int)

# Quality and condition interactions
if 'OverallQual' in combined_features_enhanced.columns and 'OverallCond' in combined_features_enhanced.columns:
    combined_features_enhanced['QualCondProduct'] = (
        combined_features_enhanced['OverallQual'] * combined_features_enhanced['OverallCond']
    )
    combined_features_enhanced['QualCondRatio'] = (
        combined_features_enhanced['OverallQual'] / (combined_features_enhanced['OverallCond'] + 1)
    )

# Room totals
room_cols = [col for col in combined_features_enhanced.columns if 'Rm' in col or 'Room' in col]
if room_cols:
    combined_features_enhanced['TotalRooms'] = combined_features_enhanced[room_cols].sum(axis=1)

# Bathroom totals
if 'FullBath' in combined_features_enhanced.columns:
    bath_cols = ['FullBath']
    if 'HalfBath' in combined_features_enhanced.columns:
        bath_cols.append('HalfBath')
    if 'BsmtFullBath' in combined_features_enhanced.columns:
        bath_cols.append('BsmtFullBath')
    if 'BsmtHalfBath' in combined_features_enhanced.columns:
        bath_cols.append('BsmtHalfBath')
    combined_features_enhanced['TotalBathrooms'] = combined_features_enhanced[bath_cols].sum(axis=1)

# Porch totals
porch_cols = [col for col in combined_features_enhanced.columns if 'Porch' in col]
if porch_cols:
    combined_features_enhanced['TotalPorchSF'] = combined_features_enhanced[porch_cols].sum(axis=1)

# Area per room
if 'TotalSF' in combined_features_enhanced.columns and 'TotalRooms' in combined_features_enhanced.columns:
    combined_features_enhanced['AreaPerRoom'] = (
        combined_features_enhanced['TotalSF'] / (combined_features_enhanced['TotalRooms'] + 1)
    )

# Lot shape features
if 'LotArea' in combined_features_enhanced.columns and 'LotFrontage' in combined_features_enhanced.columns:
    combined_features_enhanced['LotShape'] = (
        combined_features_enhanced['LotArea'] / (combined_features_enhanced['LotFrontage'] + 1)
    )

print("Enhanced features created!")
print(f"New feature count: {len(combined_features_enhanced.columns) - len(combined_features.columns)}")


In [ ]:
"""
Re-apply encoding to enhanced features
"""
# Select categorical columns from enhanced features
categorical_cols_enhanced = combined_features_enhanced.select_dtypes(include=['object']).columns
numerical_cols_enhanced = combined_features_enhanced.select_dtypes(include=np.number).columns

# Encode categorical features
if len(categorical_cols_enhanced) > 0:
    encoder_enhanced = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_encoded_enhanced = encoder_enhanced.fit_transform(combined_features_enhanced[categorical_cols_enhanced])
    X_encoded_df_enhanced = pd.DataFrame(
        X_encoded_enhanced, 
        columns=encoder_enhanced.get_feature_names_out(categorical_cols_enhanced),
        index=combined_features_enhanced.index
    )
    # Combine numerical and encoded features
    preprocessed_features_enhanced = pd.concat(
        [combined_features_enhanced[numerical_cols_enhanced], X_encoded_df_enhanced], 
        axis=1
    )
else:
    preprocessed_features_enhanced = combined_features_enhanced[numerical_cols_enhanced]

# Split back into train and test
X_train_enhanced = preprocessed_features_enhanced.iloc[:len(train_features), :]
X_test_enhanced = preprocessed_features_enhanced.iloc[len(train_features):, :]

print(f"Enhanced training data shape: {X_train_enhanced.shape}")
print(f"Enhanced test data shape: {X_test_enhanced.shape}")
print(f"Feature increase: {X_train_enhanced.shape[1] - X_train.shape[1]} new features")


In [ ]:
"""
Perform hyperparameter tuning for top performing models
"""
# Identify top 3 models from validation results
top_3_models = sorted(validation_results.items(), key=lambda x: x[1]['rmse'])[:3]
print("Top 3 models for hyperparameter tuning:")
for i, (name, metrics) in enumerate(top_3_models, 1):
    print(f"{i}. {name}: RMSE = ${metrics['rmse']:,.2f}")

print("\nStarting hyperparameter tuning (this may take a few minutes)...\n")

# Define parameter grids for common models
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'XGBoost': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 4, 5, 6],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0]
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 4, 5, 6],
        'num_leaves': [31, 50, 70],
        'subsample': [0.8, 0.9, 1.0]
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 4, 5],
        'subsample': [0.8, 0.9, 1.0]
    }
}

tuned_models = {}
kfold_tuning = KFold(n_splits=3, shuffle=True, random_state=42)  # Use 3-fold for speed

for model_name, _ in top_3_models:
    if model_name in param_grids:
        print(f"Tuning {model_name}...")
        
        # Initialize base model
        if model_name == 'Random Forest':
            base_model = RandomForestRegressor(random_state=42, n_jobs=-1)
        elif model_name == 'XGBoost':
            base_model = XGBRegressor(random_state=42, n_jobs=-1)
        elif model_name == 'LightGBM':
            base_model = LGBMRegressor(random_state=42, verbose=-1, force_col_wise=True)
        elif model_name == 'Gradient Boosting':
            base_model = GradientBoostingRegressor(random_state=42)
        
        # Perform grid search
        grid_search = GridSearchCV(
            base_model,
            param_grids[model_name],
            cv=kfold_tuning,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        grid_search.fit(X_train_enhanced, y_train)
        
        tuned_models[model_name] = grid_search.best_estimator_
        
        print(f"  Best parameters: {grid_search.best_params_}")
        print(f"  Best CV RMSE: ${-grid_search.best_score_:,.2f}\n")

print("Hyperparameter tuning completed!")

In [ ]:
"""
Create a stacking ensemble of the best models
"""
print("Creating stacking ensemble...")

# Use the top tuned models (or original if not tuned) as base estimators
base_estimators = []
for name, _ in top_3_models:
    if name in tuned_models:
        base_estimators.append((name, tuned_models[name]))
    else:
        # Use original model if not tuned
        model = models[name]
        model.fit(X_train_enhanced, y_train)  # Ensure it's fitted
        base_estimators.append((name, model))

# Create stacking regressor with Ridge as meta-learner
stacking_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=Ridge(alpha=10.0),
    cv=5,
    n_jobs=-1
)

# Train stacking model
stacking_model.fit(X_train_enhanced, y_train)

# Evaluate on validation split
X_train_enh_split, X_val_enh_split, y_train_enh_split, y_val_enh_split = train_test_split(
    X_train_enhanced, y_train, test_size=0.2, random_state=42
)

y_val_pred_stack = stacking_model.predict(X_val_enh_split)

# Calculate metrics
rmse_stack = np.sqrt(mean_squared_error(y_val_enh_split, y_val_pred_stack))
mae_stack = mean_absolute_error(y_val_enh_split, y_val_pred_stack)
r2_stack = r2_score(y_val_enh_split, y_val_pred_stack)

print("\n" + "=" * 80)
print("STACKING ENSEMBLE PERFORMANCE")
print("=" * 80)
print(f"RMSE: ${rmse_stack:,.2f}")
print(f"MAE:  ${mae_stack:,.2f}")
print(f"R²:   {r2_stack:.4f}")
print("=" * 80)

In [ ]:
"""
Create a simple weighted average ensemble
"""
print("Creating weighted average ensemble...")

# Train all tuned models on full enhanced training data
predictions = []
model_weights = {}

for name, _ in top_3_models:
    if name in tuned_models:
        model = tuned_models[name]
    else:
        model = models[name]
        model.fit(X_train_enhanced, y_train)
    
    # Get predictions on validation split
    y_pred = model.predict(X_val_enh_split)
    predictions.append(y_pred)
    
    # Calculate individual RMSE for weighting
    rmse = np.sqrt(mean_squared_error(y_val_enh_split, y_pred))
    # Weight inversely proportional to RMSE
    model_weights[name] = 1.0 / rmse

# Normalize weights
total_weight = sum(model_weights.values())
model_weights = {k: v/total_weight for k, v in model_weights.items()}

print("\nModel weights:")
for name, weight in model_weights.items():
    print(f"  {name}: {weight:.4f}")

# Create weighted average prediction
y_val_pred_weighted = np.zeros(len(y_val_enh_split))
for i, (name, _) in enumerate(top_3_models):
    y_val_pred_weighted += predictions[i] * model_weights[name]

# Calculate metrics
rmse_weighted = np.sqrt(mean_squared_error(y_val_enh_split, y_val_pred_weighted))
mae_weighted = mean_absolute_error(y_val_enh_split, y_val_pred_weighted)
r2_weighted = r2_score(y_val_enh_split, y_val_pred_weighted)

print("\n" + "=" * 80)
print("WEIGHTED AVERAGE ENSEMBLE PERFORMANCE")
print("=" * 80)
print(f"RMSE: ${rmse_weighted:,.2f}")
print(f"MAE:  ${mae_weighted:,.2f}")
print(f"R²:   {r2_weighted:.4f}")
print("=" * 80)

In [ ]:
"""
Compare all approaches and select the best final model
"""
final_comparison = {
    'Original Lasso': {
        'rmse': rmse_train,  # From original training
        'r2': r2_train
    },
    'Best Individual Model': {
        'rmse': validation_results[best_model_name]['rmse'],
        'r2': validation_results[best_model_name]['r2']
    },
    'Stacking Ensemble': {
        'rmse': rmse_stack,
        'r2': r2_stack
    },
    'Weighted Average': {
        'rmse': rmse_weighted,
        'r2': r2_weighted
    }
}

print("=" * 80)
print("FINAL MODEL COMPARISON SUMMARY")
print("=" * 80)
print(f"{'Model':<30} {'RMSE':<15} {'R²':<10}")
print("-" * 80)

for model_name, metrics in sorted(final_comparison.items(), key=lambda x: x[1]['rmse']):
    print(f"{model_name:<30} ${metrics['rmse']:>10,.2f}    {metrics['r2']:>8.4f}")

print("=" * 80)

# Determine best model
best_final_model = min(final_comparison.items(), key=lambda x: x[1]['rmse'])
print(f"\n🏆 BEST MODEL: {best_final_model[0]}")
print(f"   RMSE: ${best_final_model[1]['rmse']:,.2f}")
print(f"   R²: {best_final_model[1]['r2']:.4f}")

# Calculate improvement over baseline
improvement = ((rmse_train - best_final_model[1]['rmse']) / rmse_train) * 100
print(f"\n📈 Improvement over baseline: {improvement:.2f}%")

In [ ]:
"""
Visualize the performance improvements across all methods
"""
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# RMSE Comparison
model_names_final = list(final_comparison.keys())
rmse_values = [final_comparison[m]['rmse'] for m in model_names_final]
r2_values = [final_comparison[m]['r2'] for m in model_names_final]

colors_grad = plt.cm.RdYlGn_r(np.linspace(0.3, 0.8, len(model_names_final)))
bars1 = axes[0].bar(range(len(model_names_final)), rmse_values, color=colors_grad, alpha=0.7, edgecolor='black')
axes[0].set_xticks(range(len(model_names_final)))
axes[0].set_xticklabels(model_names_final, rotation=45, ha='right')
axes[0].set_ylabel('RMSE ($)', fontweight='bold', fontsize=12)
axes[0].set_title('Model RMSE Comparison', fontweight='bold', fontsize=14)
axes[0].grid(axis='y', alpha=0.3, linestyle='--')

for bar, val in zip(bars1, rmse_values):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'${val:,.0f}',
                ha='center', va='bottom', fontsize=9)

# R² Comparison
colors_grad_r2 = plt.cm.RdYlGn(np.linspace(0.3, 0.8, len(model_names_final)))
bars2 = axes[1].bar(range(len(model_names_final)), r2_values, color=colors_grad_r2, alpha=0.7, edgecolor='black')
axes[1].set_xticks(range(len(model_names_final)))
axes[1].set_xticklabels(model_names_final, rotation=45, ha='right')
axes[1].set_ylabel('R² Score', fontweight='bold', fontsize=12)
axes[1].set_title('Model R² Comparison', fontweight='bold', fontsize=14)
axes[1].grid(axis='y', alpha=0.3, linestyle='--')
axes[1].set_ylim([min(r2_values) - 0.01, 1.0])

for bar, val in zip(bars2, r2_values):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
"""
Generate final predictions using the best performing model
"""
print("Generating final predictions with best model...")

# Retrain best model on full enhanced training data
if best_final_model[0] == 'Stacking Ensemble':
    final_model = stacking_model
    final_model.fit(X_train_enhanced, y_train)
elif best_final_model[0] == 'Weighted Average':
    # For weighted average, we need predictions from all models
    print("Using weighted average ensemble for final predictions")
else:
    # Use best individual model
    if best_model_name in tuned_models:
        final_model = tuned_models[best_model_name]
    else:
        final_model = models[best_model_name]
    final_model.fit(X_train_enhanced, y_train)

# Generate predictions based on model type
if best_final_model[0] == 'Weighted Average':
    # Generate weighted predictions on test set
    test_predictions_final = np.zeros(len(X_test_enhanced))
    for name, _ in top_3_models:
        if name in tuned_models:
            model = tuned_models[name]
        else:
            model = models[name]
        model.fit(X_train_enhanced, y_train)
        pred = model.predict(X_test_enhanced)
        test_predictions_final += pred * model_weights[name]
else:
    test_predictions_final = final_model.predict(X_test_enhanced)

# Ensure no negative predictions
test_predictions_final = np.maximum(test_predictions_final, 0)

print(f"✓ Final predictions generated")
print(f"  Prediction range: ${test_predictions_final.min():,.2f} - ${test_predictions_final.max():,.2f}")
print(f"  Mean prediction: ${test_predictions_final.mean():,.2f}")

In [ ]:
"""
Create the final submission file with improved predictions
"""
submission_improved = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_predictions_final
})

# Save improved submission
submission_improved.to_csv('submission_improved.csv', index=False)

print("=" * 80)
print("✓ IMPROVED SUBMISSION FILE CREATED: 'submission_improved.csv'")
print("=" * 80)

# Compare with original submission
original_submission = pd.read_csv('submission.csv')
comparison = pd.DataFrame({
    'Id': test_ids,
    'Original_Prediction': original_submission['SalePrice'],
    'Improved_Prediction': test_predictions_final,
    'Difference': test_predictions_final - original_submission['SalePrice'],
    'Percent_Change': ((test_predictions_final - original_submission['SalePrice']) / 
                       original_submission['SalePrice'] * 100)
})

print("\nPrediction Comparison Statistics:")
print(f"Mean absolute difference: ${comparison['Difference'].abs().mean():,.2f}")
print(f"Mean percent change: {comparison['Percent_Change'].abs().mean():.2f}%")
print(f"Predictions increased: {(comparison['Difference'] > 0).sum()}")
print(f"Predictions decreased: {(comparison['Difference'] < 0).sum()}")
print(f"Predictions unchanged: {(comparison['Difference'] == 0).sum()}")

# Save detailed comparison
comparison.to_csv('prediction_comparison.csv', index=False)
print("\n✓ Detailed comparison saved to: 'prediction_comparison.csv'")